In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../scripts')

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import numpy as np
import glob
import numpy as np
import json
import matplotlib as mpl
import re

from plotting import plot_model_comparison

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_name = "crc" # Options: merfish, crc

In [5]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    # If model_name contains a number, then continue otherwise skip
    if not re.search(r'\d', model_name):
        continue
    holdout_celltype = "_".join(parts[2:])
    try:
        with open(fp, "r") as f:
            data = json.load(f)
    except Exception:
        # skip unreadable/invalid json
        continue

    try:
        rows.append({
            "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
            "model_name": model_name,
            "holdout_celltype": holdout_celltype,
            "n_deg": data.get("n_deg"),
            "spearman": data.get("spearman"),
            "pearson": data.get("pearson"),
            "precision": data.get("precision"),
            "direction_match": data.get("direction_match"),
            "direction_match_k": data.get("direction_match_k"),
            "direction_match_gt": data.get("direction_match_gt"),
            "mixing_index": data.get("mixing_index"),
            "edistance_global": data.get("edistance_global"),
            "edistance_local": data.get("edistance_local"),
            "edistance_pca": data.get("edistance_pca"),
            "edistance_pca_log": data.get("edistance_pca_log"),           
            "rmse": data.get("rmse"),
            "mse_lfc": data.get("mse_lfc"),
        })
    except Exception as e:
        print(f"Error processing file {fp}: {e}")
        continue

data_df = pd.DataFrame(rows)
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc
0,crc_120,cellina-W-1-cf,Endothelial_CRC,50,0.701705,0.967581,0.38,1.0,0.38,1.00,0.662835,69.904934,69.835446,157.361009,6.501936,5663.257822,0.382198
1,crc_120,cellina-W-1-cf,Epithelial_CRC,50,0.788139,0.917536,0.34,1.0,0.34,1.00,0.896148,52.027249,47.751931,300.283032,10.165348,624156.603799,5.270440
2,crc_120,cellina-W-1-cf,Fibroblast_CRC,50,0.721393,0.776356,0.32,1.0,0.32,0.88,0.653704,64.205710,62.654230,307.071874,6.999446,178565.726913,0.824445
3,crc_120,cellina-W-1-cf,Myeloid_CRC,50,0.808884,0.919488,0.40,1.0,0.40,0.96,0.885656,63.760775,62.662078,167.072000,6.473683,34467.940955,0.163949
4,crc_120,cellina-W-1-cf,T_cell_CRC,50,0.859592,0.955700,0.60,1.0,0.60,0.98,0.847765,79.505400,80.967456,154.844683,8.462617,22046.783773,0.114336


In [6]:
# Remove -cf from the end of each model_name
data_df["model_name"] = data_df["model_name"].str.replace("-cf", "", regex=False)
n_deg = data_df["n_deg"].iloc[0]
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc
0,crc_120,cellina-W-1,Endothelial_CRC,50,0.701705,0.967581,0.38,1.0,0.38,1.00,0.662835,69.904934,69.835446,157.361009,6.501936,5663.257822,0.382198
1,crc_120,cellina-W-1,Epithelial_CRC,50,0.788139,0.917536,0.34,1.0,0.34,1.00,0.896148,52.027249,47.751931,300.283032,10.165348,624156.603799,5.270440
2,crc_120,cellina-W-1,Fibroblast_CRC,50,0.721393,0.776356,0.32,1.0,0.32,0.88,0.653704,64.205710,62.654230,307.071874,6.999446,178565.726913,0.824445
3,crc_120,cellina-W-1,Myeloid_CRC,50,0.808884,0.919488,0.40,1.0,0.40,0.96,0.885656,63.760775,62.662078,167.072000,6.473683,34467.940955,0.163949
4,crc_120,cellina-W-1,T_cell_CRC,50,0.859592,0.955700,0.60,1.0,0.60,0.98,0.847765,79.505400,80.967456,154.844683,8.462617,22046.783773,0.114336


In [7]:
df = data_df.copy() # start with existing dataframe
df["sid"] = df["sid"].astype(str)

In [8]:
# Remove -W from model names
df["model_name"] = df["model_name"].str.replace("-W", "", regex=False)
df.model_name.unique()

array(['cellina-1', 'cellina-2', 'cpa-1', 'cpa-2', 'scgen-1', 'scgen-2'],
      dtype=object)

In [9]:
# Split model_name by '-' and add the last part as a new column 'seed'
df["seed"] = df["model_name"].str.split("-").str[-1]
df["model_name"] = df["model_name"].str.rsplit("-", n=1).str[0]

In [10]:
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,seed
0,crc_120,cellina,Endothelial_CRC,50,0.701705,0.967581,0.38,1.000000,0.38,1.00,0.662835,69.904934,69.835446,157.361009,6.501936,5663.257822,0.382198,1
1,crc_120,cellina,Epithelial_CRC,50,0.788139,0.917536,0.34,1.000000,0.34,1.00,0.896148,52.027249,47.751931,300.283032,10.165348,624156.603799,5.270440,1
2,crc_120,cellina,Fibroblast_CRC,50,0.721393,0.776356,0.32,1.000000,0.32,0.88,0.653704,64.205710,62.654230,307.071874,6.999446,178565.726913,0.824445,1
3,crc_120,cellina,Myeloid_CRC,50,0.808884,0.919488,0.40,1.000000,0.40,0.96,0.885656,63.760775,62.662078,167.072000,6.473683,34467.940955,0.163949,1
4,crc_120,cellina,T_cell_CRC,50,0.859592,0.955700,0.60,1.000000,0.60,0.98,0.847765,79.505400,80.967456,154.844683,8.462617,22046.783773,0.114336,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,crc_242,scgen,Endothelial_CRC,50,0.487155,0.767563,0.20,1.000000,0.20,0.92,0.847041,15.596381,17.177997,701.503215,1.679573,1899.799448,1.702493,2
176,crc_242,scgen,Epithelial_CRC,50,-0.174454,-0.159371,0.04,1.000000,0.04,0.38,0.346242,18.060975,20.894987,1050.522613,7.844713,80173.144166,17.130778,2
177,crc_242,scgen,Fibroblast_CRC,50,0.318319,0.444223,0.24,1.000000,0.24,0.68,0.799536,12.425145,14.966710,762.821425,1.222155,30971.206888,3.991282,2
178,crc_242,scgen,Myeloid_CRC,50,0.208547,0.432814,0.24,0.916667,0.22,0.74,0.545652,11.701496,14.817222,889.412552,1.580923,3395.379039,4.338654,2


In [11]:
old = pd.read_csv(f"../results/loo_summary_{dataset_name}_DEG_{n_deg}_v4.csv")

# Only keep model_name which exists in df
old = old[old["model_name"].isin(df["model_name"])]
old['seed'] = 0

In [12]:
# concat df and old
combined_df = pd.concat([df, old], ignore_index=True)

In [13]:
combined_df.model_name.unique()

array(['cellina', 'cpa', 'scgen'], dtype=object)

In [14]:
combined_df = combined_df.rename(columns={"direction_match_k": "signed_precision"})
combined_df = combined_df.rename(columns={"edistance_pca_log": "e-distance"})

In [15]:
combined_df['rmse'] = np.log10(combined_df['rmse'])
combined_df['rmse_lfc'] = np.sqrt(combined_df['mse_lfc'])

In [16]:
combined_df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,signed_precision,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,e-distance,rmse,mse_lfc,seed,nb_deviance,rmse_lfc
0,crc_120,cellina,Endothelial_CRC,50,0.701705,0.967581,0.38,1.0,0.38,1.00,0.662835,69.904934,69.835446,157.361009,6.501936,3.753066,0.382198,1,NaN,0.618221
1,crc_120,cellina,Epithelial_CRC,50,0.788139,0.917536,0.34,1.0,0.34,1.00,0.896148,52.027249,47.751931,300.283032,10.165348,5.795294,5.270440,1,NaN,2.295744
2,crc_120,cellina,Fibroblast_CRC,50,0.721393,0.776356,0.32,1.0,0.32,0.88,0.653704,64.205710,62.654230,307.071874,6.999446,5.251798,0.824445,1,NaN,0.907990
3,crc_120,cellina,Myeloid_CRC,50,0.808884,0.919488,0.40,1.0,0.40,0.96,0.885656,63.760775,62.662078,167.072000,6.473683,4.537415,0.163949,1,NaN,0.404907
4,crc_120,cellina,T_cell_CRC,50,0.859592,0.955700,0.60,1.0,0.60,0.98,0.847765,79.505400,80.967456,154.844683,8.462617,4.343345,0.114336,1,NaN,0.338135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,crc_242,scgen,Endothelial_CRC,50,0.431549,0.779002,0.28,1.0,0.28,0.90,0.806308,13.630470,15.415315,781.616081,1.456042,3.201000,1.903118,0,0.088443,1.379535
266,crc_242,scgen,Epithelial_CRC,50,-0.188379,-0.081337,0.08,1.0,0.08,0.44,0.260874,18.877773,20.680656,953.370030,7.338854,4.898389,16.403790,0,0.122120,4.050159
267,crc_242,scgen,Fibroblast_CRC,50,0.320720,0.392764,0.24,1.0,0.24,0.62,0.867351,15.095401,17.648583,814.872899,1.246071,4.480030,4.038612,0,0.086570,2.009630
268,crc_242,scgen,Myeloid_CRC,50,0.333782,0.505007,0.26,1.0,0.26,0.74,0.504493,11.114477,14.466933,996.277929,1.584157,3.536350,3.732907,0,0.083452,1.932073


In [17]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if dataset_name == "merfish":
    combined_df["holdout_celltype"] = combined_df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    combined_df["holdout_celltype"] = combined_df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [18]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
combined_df["perturbation"] = combined_df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
combined_df["holdout_celltype"] = combined_df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
combined_df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,signed_precision,direction_match_gt,...,edistance_global,edistance_local,edistance_pca,e-distance,rmse,mse_lfc,seed,nb_deviance,rmse_lfc,perturbation
0,crc_120,cellina,Endothelial,50,0.701705,0.967581,0.38,1.0,0.38,1.00,...,69.904934,69.835446,157.361009,6.501936,3.753066,0.382198,1,NaN,0.618221,CRC
1,crc_120,cellina,Epithelial,50,0.788139,0.917536,0.34,1.0,0.34,1.00,...,52.027249,47.751931,300.283032,10.165348,5.795294,5.270440,1,NaN,2.295744,CRC
2,crc_120,cellina,Fibroblast,50,0.721393,0.776356,0.32,1.0,0.32,0.88,...,64.205710,62.654230,307.071874,6.999446,5.251798,0.824445,1,NaN,0.907990,CRC
3,crc_120,cellina,Myeloid,50,0.808884,0.919488,0.40,1.0,0.40,0.96,...,63.760775,62.662078,167.072000,6.473683,4.537415,0.163949,1,NaN,0.404907,CRC
4,crc_120,cellina,T-cell,50,0.859592,0.955700,0.60,1.0,0.60,0.98,...,79.505400,80.967456,154.844683,8.462617,4.343345,0.114336,1,NaN,0.338135,CRC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,crc_242,scgen,Endothelial,50,0.431549,0.779002,0.28,1.0,0.28,0.90,...,13.630470,15.415315,781.616081,1.456042,3.201000,1.903118,0,0.088443,1.379535,CRC
266,crc_242,scgen,Epithelial,50,-0.188379,-0.081337,0.08,1.0,0.08,0.44,...,18.877773,20.680656,953.370030,7.338854,4.898389,16.403790,0,0.122120,4.050159,CRC
267,crc_242,scgen,Fibroblast,50,0.320720,0.392764,0.24,1.0,0.24,0.62,...,15.095401,17.648583,814.872899,1.246071,4.480030,4.038612,0,0.086570,2.009630,CRC
268,crc_242,scgen,Myeloid,50,0.333782,0.505007,0.26,1.0,0.26,0.74,...,11.114477,14.466933,996.277929,1.584157,3.536350,3.732907,0,0.083452,1.932073,CRC


In [19]:
metrics = ["pearson", "signed_precision", "e-distance", "rmse_lfc"]

In [20]:
# Make a df from combined_df which averages over celltype, sid and perturbation but keeps model_name and seed separate
combined_df_avg_seed = combined_df.groupby(["model_name", "seed"])[metrics].mean().reset_index()
combined_df_avg_seed = combined_df_avg_seed.round(2)
combined_df_avg_seed
# Compute mean and std of each metric for each model across seeds
combined_df_seed_summary = combined_df_avg_seed.groupby("model_name")[metrics].agg(["mean", "std"]).reset_index()
combined_df_seed_summary = combined_df_seed_summary.round(2)
combined_df_seed_summary

model_name pearson       signed_precision       e-distance       rmse_lfc  \
                mean   std             mean   std       mean   std     mean   
0    cellina    0.82  0.01             0.40  0.01       7.64  0.21     1.28   
1        cpa    0.69  0.02             0.23  0.01       6.63  0.21     1.49   
2      scgen    0.50  0.01             0.19  0.00       4.84  0.24     2.09   

         
    std  
0  0.01  
1  0.02  
2  0.09

## Seed-to-seed reliability of cellina

For each metric, decompose cellina's total variance across the 30 (slide, held-out celltype) folds × 3 seeds into a fold (subject) component and a seed (rater) component, using a two-way random-effects ANOVA. Report the intraclass correlation ICC(2,1) — absolute agreement across seeds — with a bootstrap 95% CI, plus the % of total variance attributable to seed identity. ICC(2,1) close to 1 (equivalently, % variance from seed close to 0) means fold identity explains nearly all the variability and the specific random seed used has a negligible effect. Conventionally, ICC > 0.9 is considered "excellent" reliability (Koo & Li, 2016).

In [21]:
def variance_decomposition(wide):
    """Two-way random-effects ANOVA variance decomposition and ICC(2,1).
    wide: DataFrame, rows = subjects (folds), columns = raters (seeds)."""
    X = wide.to_numpy(dtype=float)
    n, k = X.shape
    grand_mean = X.mean()
    row_means = X.mean(axis=1)
    col_means = X.mean(axis=0)

    ss_total = ((X - grand_mean) ** 2).sum()
    ss_rows = k * ((row_means - grand_mean) ** 2).sum()
    ss_cols = n * ((col_means - grand_mean) ** 2).sum()
    ss_error = ss_total - ss_rows - ss_cols

    ms_rows = ss_rows / (n - 1)
    ms_cols = ss_cols / (k - 1)
    ms_error = ss_error / ((n - 1) * (k - 1))

    icc = (ms_rows - ms_error) / (ms_rows + (k - 1) * ms_error + (k / n) * (ms_cols - ms_error))

    var_fold = max((ms_rows - ms_error) / k, 0.0)
    var_seed = max((ms_cols - ms_error) / n, 0.0)
    var_error = max(ms_error, 0.0)
    total_var = var_fold + var_seed + var_error

    return {
        "icc": icc,
        "pct_var_fold": 100 * var_fold / total_var,
        "pct_var_seed": 100 * var_seed / total_var,
        "pct_var_error": 100 * var_error / total_var,
    }

def bootstrap_icc_ci(wide, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(wide)
    values = []
    for _ in range(n_boot):
        boot = wide.iloc[rng.integers(0, n, n)]
        icc = variance_decomposition(boot)["icc"]
        if np.isfinite(icc):
            values.append(icc)
    return np.percentile(values, [2.5, 97.5])

cellina_df = combined_df[combined_df.model_name == "cellina"].copy()
cellina_df["seed"] = cellina_df["seed"].astype(str)
fold_keys = ["sid", "holdout_celltype", "perturbation"]

rows = []
for metric in metrics:
    wide = cellina_df.pivot_table(index=fold_keys, columns="seed", values=metric)
    result = variance_decomposition(wide)
    ci_low, ci_high = bootstrap_icc_ci(wide)
    rows.append({
        "metric": metric,
        "n_folds": wide.shape[0],
        "n_seeds": wide.shape[1],
        "icc": result["icc"],
        "icc_ci_low": ci_low,
        "icc_ci_high": ci_high,
        "pct_var_fold": result["pct_var_fold"],
        "pct_var_seed": result["pct_var_seed"],
        "pct_var_error": result["pct_var_error"],
    })

icc_df = pd.DataFrame(rows).set_index("metric").round(3)
icc_df

,n_folds,n_seeds,icc,icc_ci_low,icc_ci_high,pct_var_fold,pct_var_seed,pct_var_error
metric,,,,,,,,
pearson,30,3,0.856,0.710,0.930,85.572,0.000,14.428
signed_precision,30,3,0.903,0.786,0.954,90.147,0.000,9.853
e-distance,30,3,0.777,0.653,0.866,77.723,1.825,20.453
rmse_lfc,30,3,0.973,0.961,0.984,97.227,0.000,2.773
